# KGSS 오염 프로브 — 로짓 + 생성 (Colab)

`kgss_contamination.py build`로 로컬에서 만든 `probe/probes.csv`를 EXAONE 3.5 / Kanana 1.5에 돌린다.
- **A(로짓분포)**: 첫 토큰 로짓만 뽑음 (forward pass 1회, 빠름) → `logit_responses.csv`
- **B(원문완성)/C(선택지순서)**: 실제로 텍스트를 생성함 (모델당 60건) → `responses.csv`

**실행 전에**: 상단 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU`로 설정.

**필요한 파일 2개** (아래 업로드 셀에서 직접 선택):
- `kgss_token_check.py`
- `probe/probes.csv`

**끝나면**: `probe/logit_responses.csv`와 `probe/responses.csv`를 다운받아 로컬 프로젝트의 `probe/` 폴더에 넣고
```powershell
uv run kgss_contamination.py score --out .\probe
```
를 실행하면 된다.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 패키지 설치

**주의**: `torch`/`torchvision`은 Colab 기본 제공 버전을 그대로 쓴다. 과거 함정(CLAUDE.md 6절) — 이것들을 임의로 올리거나 내리면 버전 충돌이 난다.

**`transformers`는 `-U`(최신)가 아니라 `4.48.3`으로 고정한다.** 최신 transformers(5.x대, `huggingface_hub.dataclasses`의 `@strict` 검증)는 `LlamaConfig.validate_architecture()`가 `head_dim`이 명시돼 있는지와 무관하게 `hidden_size % num_attention_heads != 0`이면 무조건 에러를 낸다. Kanana는 `hidden_size=1792`, `num_attention_heads=24`라 나누어떨어지지 않는데, `head_dim=128`이 명시돼 있어 실제로는 유효한 구성이다 — 최신 transformers 쪽의 과도하게 엄격한 검사다. `4.48.3`은 Kanana `config.json`에 기록된 검증 버전이라 이 문제가 없다. **(2026-09-04 확인: Kanana·EXAONE 둘 다 이 고정으로 정상 로드됨.)**

`diffusers`/`gradio`가 `huggingface_hub` 버전 충돌을 경고할 수 있는데, 둘 다 이 노트북에서 안 쓰므로 무시해도 된다.

**EXAONE은 반대로 최신 API를 요구한다.** 저장소가 2026-02에 transformers v5 대응 커밋을 올려서, 리비전을 안 박으면 최신 원격 코드를 받아와 4.48.3과 안 맞는다(`ImportError: cannot import name 'RopeParameters'`). 3번 셀에서 `--revision 8e6fc27`(v5 대응 이전 최초 커밋)로 고정해 우회한다.

In [ ]:
!pip install -q "transformers==4.48.3" accelerate

## 2. 파일 업로드

`kgss_token_check.py`와 `probes.csv`를 함께 선택해서 올린다. 이전 세션에서 받아둔 `logit_responses.csv`/`responses.csv`가 있으면 같이 올려도 된다 — 이번에 실행한 모델의 행만 교체되고 나머지는 보존된다.

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

Path("probe").mkdir(exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    if name == "kgss_token_check.py":
        continue  # 루트에 그대로 둠
    if name in ("probes.csv", "logit_responses.csv", "responses.csv") or any(
            name.endswith("/" + n) for n in ("probes.csv", "logit_responses.csv", "responses.csv")):
        target = name.split("/")[-1]
        shutil.move(name, f"probe/{target}")

print("루트:", sorted(Path(".").glob("*.py")))
print("probe/:", sorted(Path("probe").glob("*")))
assert Path("kgss_token_check.py").exists(), "kgss_token_check.py가 없습니다"
assert Path("probe/probes.csv").exists(), "probe/probes.csv가 없습니다"

## 3. 로짓 추출 + B/C 생성 실행

EXAONE 3.5(2.4B)와 Kanana 1.5(2.1B)를 순서대로 로드해 A(로짓분포)를 채점하고, 이어서 같은 모델로 B(원문완성)/C(선택지순서) 60건을 그리디 디코딩으로 생성한다(`--no-contam-generate`로 끌 수 있음). 한 모델이 로드에 실패해도 스크립트는 멈추지 않고 다음 모델로 넘어간다.

**`--revision`은 EXAONE에 필요한 값(`8e6fc27`)이라 Kanana에도 같이 적용된다.** Kanana 저장소엔 그 해시가 없어 실패할 수 있으니, 아래처럼 **모델별로 따로 실행**한다 — `kgss_token_check.py`가 이번에 실행한 모델(`--models`)의 결과만 `logit_responses.csv`/`responses.csv`에서 교체하고 나머지는 보존하도록 되어 있어 순서 상관없이 안전하다.

In [ ]:
!python kgss_token_check.py \
  --models kakaocorp/kanana-1.5-2.1b-instruct-2505 \
  --contam-probes probe/probes.csv \
  --out probe \
  --device cuda

In [ ]:
!python kgss_token_check.py \
  --models LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct \
  --revision 8e6fc27 \
  --contam-probes probe/probes.csv \
  --out probe \
  --device cuda

## 4. 결과 확인

두 모델 다 `model` 열에 잡혔는지, `mass`(A, 선택지 토큰에 실린 확률질량)가 너무 낮은 행이 없는지, B/C 응답이 비어있지 않은지 채점 전에 훑어본다.

In [ ]:
import pandas as pd

lg = pd.read_csv("probe/logit_responses.csv")
print(f"A(로짓) 총 {len(lg)}행")
print(lg.groupby("model").size())
print("
mass 분포:")
print(lg.groupby("model")["mass"].describe())

resp = pd.read_csv("probe/responses.csv")
resp["len"] = resp["response"].astype(str).str.len()
print(f"
B/C(생성) 총 {len(resp)}행")
print(resp.groupby("model").size())
print("
응답 길이(문자 수) 분포:")
print(resp.groupby("model")["len"].describe())
resp.head()

## 5. 다운로드

`logit_responses.csv`와 `responses.csv`는 로컬 프로젝트의 `probe/` 폴더에 그대로 덮어쓰면 된다. 토큰화 진단 `.md` 파일들도 참고용으로 함께 받는다.

In [ ]:
from google.colab import files
from pathlib import Path

files.download("probe/logit_responses.csv")
files.download("probe/responses.csv")
for md in Path(".").glob("*.md"):
    files.download(str(md))